In [14]:
# import pandas as pd
import pandas as pd
import numpy as np

Loading the Data set 

In [15]:
sessions = pd.read_csv("datasets/stadiochoice/table1_trial_viewer_behaviour.csv")
billing = pd.read_csv("datasets/stadiochoice/table2_subscription_billing.csv")
outcome = pd.read_csv("datasets/stadiochoice/table3_conversion_outcome.csv")
demographics = pd.read_csv("datasets/stadiochoice/table4_trial_user_demographics.csv")


In [16]:
# Inspect the sessions  data set
print("sessions shape:", sessions.shape)
print("sessions duplicate count:", sessions.duplicated().sum())
print("sessions head:")
display(sessions.head())

sessions shape: (123470, 12)
sessions duplicate count: 369
sessions head:


,account_id,session_id,session_start,trial_start_date,main_device,genre_watched_per_session,search_query_text,session_watch_time,total_watch_time,genres_watched_list,searches_made,no_results_searches
0,ACC-4000000,SESS-000000,2024-11-28 13:15,2024-11-20,Smart TV,Reality,big brother,48.9,1826.0,Reality,46,7
1,ACC-4000000,SESS-000001,2024-12-01 14:36,2024-11-20,Smart TV,Reality,NaN,33.8,1826.0,Reality,46,7
2,ACC-4000000,SESS-000002,2024-11-24 02:09,2024-11-20,Smart TV,Reality,NaN,81.3,1826.0,Reality,46,7
3,ACC-4000000,SESS-000003,2024-11-27 02:10,2024-11-20,Smart TV,Reality,NaN,48.4,1826.0,Reality,46,7
4,ACC-4000000,SESS-000004,2024-11-28 23:28,2024-11-20,Smart TV,Reality,reality show; cooking competition,35.1,1826.0,Reality,46,7


In [17]:
# Inspect the data set
print("billing shape:", billing.shape)
print("billing duplicate count:", billing.duplicated().sum())
print("billing head:")
display(billing.head())

billing shape: (18756, 6)
billing duplicate count: 56
billing head:


,account_id,payment_plan_filled_in,sign_up_platform,region,referral_or_promotion_code,trial_plan_chosen
0,ACC-4000000,True,Promotion Ad,KZN,True,Basic
1,ACC-4000003,True,App,GP,False,Premium
2,ACC-4000006,True,Website,WC,False,Standard
3,ACC-4000008,False,Website,GP,False,Premium
4,ACC-4000010,True,App,EC,False,Premium


In [18]:
# Inspect the data set
print("outcome shape:", outcome.shape)
print("outcome duplicate count:", outcome.duplicated().sum())
print("outcome head:")
display(outcome.head())

outcome shape: (18756, 4)
outcome duplicate count: 56
outcome head:


,account_id,converted_to_paid_subscription,date_of_conversion,early_churn
0,ACC-4000000,True,2024-12-02,False
1,ACC-4000003,True,2025-05-27,False
2,ACC-4000006,True,2025-02-13,False
3,ACC-4000008,True,2025-03-31,False
4,ACC-4000010,False,NaN,NaN


In [19]:
# Inspect the demographics data set
print("demographics:", demographics.shape)
print("demographics duplicate count:", demographics.duplicated().sum())
print("demographics head:")
display(demographics.head())

demographics: (18756, 3)
demographics duplicate count: 56
demographics head:


,account_id,age_band,stated_interests
0,ACC-4000000,25-34,"News, Comedy, Sport"
1,ACC-4000003,35-44,"Movies, News, Local content, Kids"
2,ACC-4000006,25-34,"Reality, Documentaries"
3,ACC-4000008,25-34,"Sport, News"
4,ACC-4000010,55+,"Reality, Comedy, Kids"


In [20]:
print("sessions unique accounts:", sessions['account_id'].nunique())
print("billing unique accounts:", billing['account_id'].nunique())
print("outcome unique accounts:", outcome['account_id'].nunique())
print("demographics unique accounts:", demographics['account_id'].nunique())

print("\nConversion counts:")
print(outcome['converted_to_paid_subscription'].value_counts())

print("\nConversion proportions (normalise=True):")
print(outcome['converted_to_paid_subscription'].value_counts(normalize=True))


sessions unique accounts: 18700
billing unique accounts: 18700
outcome unique accounts: 18700
demographics unique accounts: 18700

Conversion counts:
converted_to_paid_subscription
False    9912
True     8844
Name: count, dtype: int64

Conversion proportions (normalise=True):
converted_to_paid_subscription
False    0.528471
True     0.471529
Name: proportion, dtype: float64



# The loaded data show:
 **Duplicates:** There were 123 470 session rows which covered 18700 unquie accounts while the billing, outcome and demographics tables have 18756 rows. There were 369 duplicate rows in sessions and 56 in each of the other three. I will remove this once im in the preprocessing phase.
 
 **Classes and counts:**  The target column (converted to paid subscription) has two classes "true" for the users who converted to paid and False for the ones who didnt. So 9912 False and 8844 True 

 **Imbalance:** : The target split is around 47% True and 53% False which is pretty balanced which thne makes it easier to work with compared to SS2 where there was 11% positive cases.

 **Why the four separate tables:** Ive kept them separate for nor because the "sessions" records every single stream, so one user would have mutliple rows. Then the other three tables have one row per account. Having them seperate will help me clean before merging.
 

PreProcessing the Data set

In [21]:
# removing the duplicates from the 4 tables
print("sessions shape before:", sessions.shape)
sessions = sessions.drop_duplicates()
print("sessions shape after:", sessions.shape)

print("billing shape before:", billing.shape)
billing = billing.drop_duplicates()
print("billing shape after:", billing.shape)

print("outcome shape before:", outcome.shape)
outcome = outcome.drop_duplicates()
print("outcome shape after:", outcome.shape)

print("demographics shape before:", demographics.shape)
demographics = demographics.drop_duplicates()
print("demographics shape after:", demographics.shape)

# check the repeated sessionID values
print("Duplicate session_id count:", sessions['session_id'].duplicated().sum())

sessions shape before: (123470, 12)
sessions shape after: (123101, 12)
billing shape before: (18756, 6)
billing shape after: (18700, 6)
outcome shape before: (18756, 4)
outcome shape after: (18700, 4)
demographics shape before: (18756, 3)
demographics shape after: (18700, 3)
Duplicate session_id count: 0


### Firstly removed the duplicates

**What I removed:** I removed the 369 dupicate rows that were in the "sessions" table, 56 duplicate rows from the account tables for billing, outcome and demographics.
**Reason why I removed:** If there duplicates were not removed, there would be a high chance of double counting and not giving the true reflection in numbers like the total watch time which can lead to a data leakage by having the same account in the training and test data sets. 
**How I checked it worked:** After cleaning the data by removing, the billing, outcome and demographics all ended up on 18700 rows which then matched the 18700 unique accounts. I also checked for repeated session Ids and the results were then zero, which shows that there werent sessions show twice in the table. 

In [22]:
# checking the missing values in the tables in the tables 
print("sessions missing values:")
print(sessions.isna().sum())

print("\nbilling missing values:")
print(billing.isna().sum())

print("\noutcome missing values:")
print(outcome.isna().sum())

print("\ndemographics missing values:")
print(demographics.isna().sum())

sessions missing values:
account_id                       0
session_id                       0
session_start                    0
trial_start_date                 0
main_device                      0
genre_watched_per_session        0
search_query_text            56534
session_watch_time               0
total_watch_time                 0
genres_watched_list              0
searches_made                    0
no_results_searches              0
dtype: int64

billing missing values:
account_id                    0
payment_plan_filled_in        0
sign_up_platform              0
region                        0
referral_or_promotion_code    0
trial_plan_chosen             0
dtype: int64

outcome missing values:
account_id                           0
converted_to_paid_subscription       0
date_of_conversion                9885
early_churn                       9885
dtype: int64

demographics missing values:
account_id             0
age_band            1127
stated_interests    1119
dtype: int64


In [23]:
print(outcome.isna().sum())

account_id                           0
converted_to_paid_subscription       0
date_of_conversion                9885
early_churn                       9885
dtype: int64


In [24]:
# check that blank the conversion dates are the non converted users
print(outcome['converted_to_paid_subscription'].value_counts())
print(pd.crosstab(outcome['converted_to_paid_subscription'], outcome['date_of_conversion'].isna()))

converted_to_paid_subscription
False    9885
True     8815
Name: count, dtype: int64
date_of_conversion              False  True 
converted_to_paid_subscription              
False                               0   9885
True                             8815      0


**shows new target counts after the duplicatation removal:** So now 9885 False and 8815 True but the balance is still similar

### In finding missing values 
**What I found:** There are blank search query texts in the "sessions" table, a blank "age_band"and "stated_interests" in the "Demographics" table. The blank conversion dates and early churcn were in the "outcome" table.
**What the blanks mean:** A blank search text indicates there was no search made during the session. The blank conversion dates and early churn values meant the accounts never converted to a paid subscription, which shows in in the True False crosstab. Its only the demographics blanks, that are age band and stated interests are the actual missing value data.
**What I'll do:** Instead of not using and throwing away those accounts, Ill fill in the demographics blanks as "unknown", then with the outcome table, ill drop the conversion date and early churn columns before the modelling process anyway because that can leak the answer. 


In [25]:
# the sample of 10 rows of session start and trial start date to check date formats
print(sessions[['session_start', 'trial_start_date']].sample(10, random_state=1))

           session_start trial_start_date
41057   2025-10-29 20:52       2025-10-22
95653   2025-04-18 05:10       2025-04-08
15663   2024-08-30 03:05       2024-08-28
16520   2026-04-21 23:52       2026-04-14
22996   2024-09-06 08:05       2024-09-04
112794  2025-03-28 16:23       2025-03-27
49630   2026-05-25 17:30       2026-05-11
7751    2025-04-24 01:38       2025-04-10
64259   2025-11-12 16:39       2025-11-04
67290   2026-05-23 18:17       2026-05-22


In [27]:
# checking the failures for session starts
parsed_start = pd.to_datetime(sessions['session_start'], errors='coerce')
print("session_start failed count:", parsed_start.isna().sum())
print("session_start failed samples:")
print(sessions.loc[parsed_start.isna(), 'session_start'].head(10))

# checking the failures for trial start date
parsed_trial = pd.to_datetime(sessions['trial_start_date'], errors='coerce')
print("\ntrial_start_date failed count:", parsed_trial.isna().sum())
print("trial_start_date failed samples:")
print(sessions.loc[parsed_trial.isna(), 'trial_start_date'].head(10))

session_start failed count: 1277
session_start failed samples:
258    2026/04/13 16:37
264    2024/10/31 07:11
369    2025/09/08 11:44
432    12/05/2025 06:03
577    2026/02/21 09:26
765    2025/11/21 05:17
829    11/11/2024 05:25
896    2025/09/22 10:45
900    2025/09/22 11:31
905    08/04/2025 09:07
Name: session_start, dtype: str

trial_start_date failed count: 1238
trial_start_date failed samples:
61      2025/05/10
87     09-Jul-2025
216    24-Oct-2024
241     2025/09/21
264     2024/10/28
421     2024/10/24
495     21/01/2025
590    18-Feb-2026
811     14/02/2026
938    26-Jul-2024
Name: trial_start_date, dtype: str


In [30]:
# test if the session start format is day first (dd/mm/yyyy)
parsed_start = pd.to_datetime(sessions['session_start'], errors='coerce')
failed_sessions = sessions.loc[parsed_start.isna(), 'session_start']

# then only keep the ones matching the format of dd/mm/yyyy at the start
filtered_failed = failed_sessions[failed_sessions.str.match(r'^\d{2}/\d{2}/\d{4}')]

# split the '/' to change to numbers
split_parts =filtered_failed.str.split('/', expand=True)
part0 =pd.to_numeric(split_parts[0])

# if there are numbers in the first part greater than 12, it will be the day
print("Count where first part is greater than 12:", (part0 > 12).sum())

Count where first part is greater than 12: 386


In [33]:
# check the 2nd part of the session start
parsed_start = pd.to_datetime(sessions['session_start'], errors='coerce')
failed_sessions = sessions.loc[parsed_start.isna(), 'session_start']
filtered_failed = failed_sessions[failed_sessions.str.match(r'^\d{2}/\d{2}/\d{4}')]
split_start = filtered_failed.str.split('/', expand=True)
part1 = pd.to_numeric(split_start[1])

print("session_start second part > 12 count:", (part1 > 12).sum())

# check the trial start date part
parsed_trial = pd.to_datetime(sessions['trial_start_date'], errors='coerce')
failed_trial = sessions.loc[parsed_trial.isna(), 'trial_start_date']
filtered_failed_trial = failed_trial[failed_trial.str.match(r'^\d{2}/\d{2}/\d{4}')]
split_trial = filtered_failed_trial.str.split('/', expand=True)
part0_trial = pd.to_numeric(split_trial[0])
part1_trial = pd.to_numeric(split_trial[1])

print("trial_start_date first part > 12 count:", (part0_trial > 12).sum())
print("trial_start_date second part > 12 count:", (part1_trial > 12).sum())



session_start second part > 12 count: 0
trial_start_date first part > 12 count: 254
trial_start_date second part > 12 count: 0


In [36]:
# parse the session start by the three formats
parsed_a = pd.to_datetime(sessions['session_start'], format='%Y-%m-%d %H:%M', errors='coerce')
parsed_b = pd.to_datetime(sessions['session_start'], format='%Y/%m/%d %H:%M', errors='coerce')
parsed_c = pd.to_datetime(sessions['session_start'], format='%d/%m/%Y %H:%M', errors='coerce')

# combine them by filling in the blanks
session_start_clean = parsed_a.fillna(parsed_b).fillna(parsed_c)

# check how many were not able to parse
print(session_start_clean.isna().sum())

0


In [39]:
# parse trial start date witht he four formats
parsed_a2 = pd.to_datetime(sessions['trial_start_date'], format='%Y-%m-%d', errors='coerce')
parsed_b2 = pd.to_datetime(sessions['trial_start_date'], format='%Y/%m/%d', errors='coerce')
parsed_c2 = pd.to_datetime(sessions['trial_start_date'], format='%d-%b-%Y', errors='coerce')
parsed_d2 = pd.to_datetime(sessions['trial_start_date'], format='%d/%m/%Y', errors='coerce')  
                                                         
# chain the fills seqquentially
trial_start_clean = parsed_a2.fillna(parsed_b2).fillna(parsed_c2).fillna(parsed_d2)

# check the rest of the missing values
print(trial_start_clean.isna().sum())

0


In [40]:
# calculate the full days between the trial start and the session start
days_into_trial = (session_start_clean.dt.normalize() - trial_start_clean).dt.days

# show the minimun and maxi days
print("Minimum days into trial:", days_into_trial.min())
print("Maximum days into trial:", days_into_trial.max())

# show the numbers that are lower than 0 and the ones above 14
print("Count below 0 (negative):", (days_into_trial < 0).sum())
print("Count above 14:", (days_into_trial > 14).sum())


Minimum days into trial: 0
Maximum days into trial: 14
Count below 0 (negative): 0
Count above 14: 0


In [41]:
# overwriting the orginial columns witht he cleaned date time 
sessions['session_start'] = session_start_clean
sessions['trial_start_date'] = trial_start_clean

# show the data types to confirm they now in datetime64
print("Data Types")
print(sessions[['session_start', 'trial_start_date']].dtypes)

# print out the first few rows to show results are coming out right
print("\nHead of cleaned date columns:")
print(sessions[['session_start', 'trial_start_date']].head())

Data Types
session_start       datetime64[us]
trial_start_date    datetime64[us]
dtype: object

Head of cleaned date columns:
        session_start trial_start_date
0 2024-11-28 13:15:00       2024-11-20
1 2024-12-01 14:36:00       2024-11-20
2 2024-11-24 02:09:00       2024-11-20
3 2024-11-27 02:10:00       2024-11-20
4 2024-11-28 23:28:00       2024-11-20


### cleaning the date structure and having a standard format
**What I found:** Both the session strat and trial start date were saved as text strings with mixed formats for example the standard dashed, year first format, date first format, month as text etc. Default parsing would have not worked in the background a many rows without flagging the missing values 'NaN' and having silent data errors.
**What I Did:** Changed the known formats by using the pyhton format codes of '%Y-%m-%d', '%d/%m/%Y' and '%d-%b-%Y' then chained by '.fillna()' calls so they merge cleanly.
- the 'session_start' formats:'%Y-%m-%d %H:%M', '%Y/%m/%d %H:%M',. and '%d/%m/%Y %H:%M'
- the 'trial_start_date' formats: '%Y-%m-%d', '%Y/%m/%d', '%d-%b-%Y', and '%d/%m/%Y'
**How I checked it worked** Confirmed by showing there werent any missing values left, so the results should have been 0 on both columns. Another way I confirmed was by caluculating the time difference between the session start and the trial start, which showed 100% of the ssions fell corretly within the 0 to 14 days trial window.



In [42]:
### The Genre spelling

# show the number of unique values
print("Number of unique genres:", sessions['genre_watched_per_session'].nunique())

# Print out the top 40 value counts to check for messy spellings or variations in them
print("\nGenre Value Counts (Top 40):")
print(sessions['genre_watched_per_session'].value_counts().head(40))

Number of unique genres: 90

Genre Value Counts (Top 40):
genre_watched_per_session
Sport           8481
Soapies         8452
Movies          8341
Thriller        8123
News            8110
Action          8094
Reality         8089
Music           8027
Romance         7969
Local Drama     7868
Lifestyle       7863
Kids            7859
Documentary     7833
Comedy          7827
Drama           7708
Thriller          44
soapies           44
SOAPIES           42
Sport             42
Documentary       41
 Comedy           41
REALITY           41
MUSIC             41
  Thriller        41
DOCUMENTARY       40
 Documentary      40
  Movies          40
music             39
 Music            39
THRILLER          39
LOCAL DRAMA       38
  Action          38
thriller          37
  Romance         37
 News             37
  Reality         36
 Kids             36
 Sport            36
  Lifestyle       36
ROMANCE           36
Name: count, dtype: int64


In [43]:
# clean the genres by stripping whitespace, and converting to lowercase for consistency
sessions['genre_watched_per_session'] = sessions['genre_watched_per_session'].str.strip().str.title()

# confirm the new unique count and value counts
print("Number of unique genres after cleaning:", sessions['genre_watched_per_session'].nunique())
print("\nFull Genre Value Counts:")
print(sessions['genre_watched_per_session'].value_counts())

Number of unique genres after cleaning: 15

Full Genre Value Counts:
genre_watched_per_session
Sport          8647
Soapies        8624
Movies         8504
Thriller       8308
News           8270
Action         8252
Reality        8249
Music          8200
Romance        8134
Local Drama    8025
Kids           8023
Lifestyle      8012
Documentary    8009
Comedy         7991
Drama          7853
Name: count, dtype: int64


In [44]:
# check the main device
print("=== main_device ===")
print(sessions['main_device'].value_counts().head(20))
print("\n")

# check sign up platform 
print("=== sign_up_platform ===")
print(billing['sign_up_platform'].value_counts().head(20))
print("\n")

# check the region
print("=== region ===")
print(billing['region'].value_counts().head(20))
print("\n")

# check trial plan chosen 
print("=== trial_plan_chosen ===")
print(billing['trial_plan_chosen'].value_counts().head(20))
print("\n")

# check payment plan filled in
print("=== payment_plan_filled_in ===")
print(billing['payment_plan_filled_in'].value_counts().head(20))
print("\n")

# check age band
print("=== age_band ===")
print(demographics['age_band'].value_counts().head(20))

=== main_device ===
main_device
Mobile        45503
Smart TV      42073
Web           21233
Tablet        11964
  Mobile        190
mobile          179
Mobile          179
MOBILE          176
Smart TV        170
smart tv        164
 Mobile         163
SMART TV        161
  Smart TV      141
 Smart TV       140
  Web            97
web              89
WEB              89
Web              78
 Web             74
tablet           57
Name: count, dtype: int64


=== sign_up_platform ===
sign_up_platform
App               7446
Website           5433
Promotion Ad      3380
Referral          2029
app                 44
App                 39
 App                38
APP                 33
  App               28
WEBSITE             26
Website             24
  Website           24
 Website            23
website             21
Promotion Ad        17
  Promotion Ad      16
PROMOTION AD        15
 Promotion Ad       13
promotion ad        12
referral             8
Name: count, dtype: int64


=== region

In [45]:
# 1. Clean main_device (strip, title case, and fix Smart Tv to Smart TV)
sessions['main_device'] = sessions['main_device'].str.strip().str.title().replace('Smart Tv', 'Smart TV')

# 2. Clean sign_up_platform
billing['sign_up_platform'] = billing['sign_up_platform'].str.strip().str.title()

# 3. Clean trial_plan_chosen
billing['trial_plan_chosen'] = billing['trial_plan_chosen'].str.strip().str.title()

# 4. Clean region (uppercase since they are province codes)
billing['region'] = billing['region'].str.strip().str.upper()

# 5. Clean age_band (strip whitespace only)
demographics['age_band'] = demographics['age_band'].str.strip()

# Print full value counts to verify
print("=== main_device ===")
print(sessions['main_device'].value_counts())
print("\n=== sign_up_platform ===")
print(billing['sign_up_platform'].value_counts())
print("\n=== trial_plan_chosen ===")
print(billing['trial_plan_chosen'].value_counts())
print("\n=== region ===")
print(billing['region'].value_counts())
print("\n=== age_band ===")
print(demographics['age_band'].value_counts())

=== main_device ===
main_device
Mobile      46390
Smart TV    42849
Web         21660
Tablet      12202
Name: count, dtype: int64

=== sign_up_platform ===
sign_up_platform
App             7628
Website         5551
Promotion Ad    3453
Referral        2068
Name: count, dtype: int64

=== trial_plan_chosen ===
trial_plan_chosen
Standard    8400
Basic       5652
Premium     4648
Name: count, dtype: int64

=== region ===
region
GP     6127
KZN    3626
WC     2819
EC     1439
MP     1344
LP     1134
NW      934
FS      895
NC      382
Name: count, dtype: int64

=== age_band ===
age_band
25-34    5464
35-44    4119
18-24    3295
45-54    2610
55+      2085
Name: count, dtype: int64


### Category texts cleaning and standardising

**What I found:** There were data entry incosistencies in the categorlical columns, which lead to columns like genre per session having 90 unqiue variants which was caused by the lower and uppercase incosistency and the hidde leading trailing spaces. There were similar casing and spacing in the device, platform, plan and region columns.
**What I did:** Standardised the text columns by the string methods
- .str.strip() to remove the whitespace
- .str.title() to enforce consistent title cases in the genre formats. The device, sign up platforms and trial plans(by a specific override to clean up "Smart Tv" to Smart TV)
- .str.upper() for the regions so that the SA codes stay as "GP", "KZN", "WC"etc.
- left the age band with just .str.strip() since the numbers dont need casing

*How I checked:** Ran the full value counts() on all the cleaned columns, to see if there they went to their proper columns 





In [50]:
# check the summary of the statistics and the large values for the watch time 
print(sessions['session_watch_time'].describe())
print("\nTop 10 largest watch times:")
print(sessions['session_watch_time'].nlargest(10))

print((sessions['session_watch_time'] > 240).sum())


count    123101.000000
mean         31.034668
std          30.236045
min           0.600000
25%          12.700000
50%          22.200000
75%          38.700000
max         763.600000
Name: session_watch_time, dtype: float64

Top 10 largest watch times:
10177     763.6
122010    634.3
78658     609.5
39212     594.2
122520    567.2
40684     536.0
69040     535.1
7357      533.3
45401     510.0
51792     509.4
Name: session_watch_time, dtype: float64
214


### Session watch time outlier check
They range from 0.6 to 763.6 minutes and 22 median minutes with 214 sessions running over 4 hours. So ill be keeping them since they are under 24 hours and the 214 is a time fraction that wont affect the Random Forest model

Training the Model

Validating the Model

Results as Visuals